# **1. Perkenalan Dataset**

Tahap pertama, Anda harus mencari dan menggunakan dataset dengan ketentuan sebagai berikut:

1. **Sumber Dataset**:  
   Dataset dapat diperoleh dari berbagai sumber, seperti public repositories (*Kaggle*, *UCI ML Repository*, *Open Data*) atau data primer yang Anda kumpulkan sendiri.


**Dataset yang digunakan: UCI Wine Quality (White Wine).**

Dataset ini berasal dari [UCI Machine Learning Repository - Wine Quality](https://archive.ics.uci.edu/ml/datasets/wine+quality). Dataset memuat **4,898 baris** dan **12 kolom** (11 fitur + 1 target) dengan header, dengan pemisah semicolon (;).

Fitur-fiturnya meliputi: `fixed acidity`, `volatile acidity`, `citric acid`, `residual sugar`, `chlorides`, `free sulfur dioxide`, `total sulfur dioxide`, `density`, `pH`, `sulphates`, `alcohol`, dan `quality`.

Kolom `quality` aslinya bernilai 0-10 (skor kualitas wine).

Untuk pengerjaan submission ini, target dibinarisasi menjadi **klasifikasi biner**: `quality >= 6` → 1 (good), dan `< 6` → 0 (bad).

# **2. Import Library**

Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning atau deep learning.

In [ ]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

def _find_preprocessing_dir() -> Path:
    for candidate in (Path.cwd(), Path.cwd() / "preprocessing"):
        if (candidate / "wine-quality-white_raw.csv").exists():
            return candidate.resolve()
    # Jika tidak ada, coba cek parent
    parent = Path.cwd().parent
    if (parent / "wine-quality-white_raw.csv").exists():
        return Path.cwd().resolve()
    raise FileNotFoundError(
        "tidak dapat menemukan direktori preprocessing yang berisi "
        "'wine-quality-white_raw.csv' (jalankan notebook dari folder preprocessing/)"
    )


PREPROCESSING_DIR = _find_preprocessing_dir()
RAW_PATH = PREPROCESSING_DIR.parent / "wine-quality-white_raw.csv"
OUTPUT_PATH = PREPROCESSING_DIR / "wine-quality-white_preprocessing.csv"

_automate_path = PREPROCESSING_DIR / "automate_Devani.py"
_spec = importlib.util.spec_from_file_location("automate_preprocessing", _automate_path)
automate = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(automate)

# Pakai kembali konstanta dan fungsi preprocessing.
DATASET_NAME = automate.DATASET_NAME
TARGET_COLUMN = automate.TARGET_COLUMN
preprocess_dataset = automate.preprocess_dataset

print("Pandas            :", pd.__version__)
print("NumPy             :", np.__version__)
print("Direktori data    :", PREPROCESSING_DIR)
print("Dataset name      :", DATASET_NAME)
print("Kolom target      :", TARGET_COLUMN)

# **3. Memuat Dataset**

Pada tahap ini, Anda perlu memuat dataset ke dalam notebook. Jika dataset dalam format CSV, Anda bisa menggunakan pustaka pandas untuk membacanya. Pastikan untuk mengecek beberapa baris awal dataset untuk memahami strukturnya dan memastikan data telah dimuat dengan benar.

Jika dataset berada di Google Drive, pastikan Anda menghubungkan Google Drive ke Colab terlebih dahulu. Setelah dataset berhasil dimuat, langkah berikutnya adalah memeriksa kesesuaian data dan siap untuk dianalisis lebih lanjut.

Jika dataset berupa unstructured data, silakan sesuaikan dengan format seperti kelas Machine Learning Pengembangan atau Machine Learning Terapan

In [ ]:
# Baca file mentah Wine Quality dengan delimiter semicolon dan header.
try:
    df_raw = pd.read_csv(RAW_PATH, sep=";")
except Exception as exc: 
    raise RuntimeError(
        f"dataset could not be loaded from {RAW_PATH!r}: {exc}"
    ) from exc

n_rows, n_cols = df_raw.shape
print(f"Dataset berhasil dimuat: {n_rows} baris x {n_cols} kolom")
df_raw.head()

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, Anda akan melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.

Tujuan dari EDA adalah untuk memperoleh wawasan awal yang mendalam mengenai data dan menentukan langkah selanjutnya dalam analisis atau pemodelan.

**4.1 Struktur dan tipe data.** Periksa jumlah baris/kolom serta tipe data tiap fitur.

In [ ]:
# Jumlah baris/kolom dan tipe data setiap fitur.
print(f"Jumlah baris : {df_raw.shape[0]}")
print(f"Jumlah kolom : {df_raw.shape[1]}")
print("\nTipe data per fitur:")
print(df_raw.dtypes)
print("\nInfo dataset:")
df_raw.info()

**4.2 Ringkasan statistik numerik** untuk tiap fitur.

In [ ]:
# Ringkasan statistik numerik (count, mean, std, min, 25/50/75%, max).
df_raw.describe()

**4.3 Pemeriksaan missing value**

In [ ]:
# Cek missing values
missing_counts = df_raw.isna().sum()
print("Jumlah nilai hilang per fitur:")
print(missing_counts)
print(f"\nTotal missing values: {missing_counts.sum()}")

**4.4 Pemeriksaan duplikat**

In [ ]:
# Cek data duplikat
n_duplicates = df_raw.duplicated().sum()
print(f"Jumlah baris duplikat: {n_duplicates}")
print(f"Persentase duplikat: {n_duplicates / len(df_raw) * 100:.2f}%")

**4.5 Visualisasi**: distribusi target

In [ ]:
# Visualisasi distribusi target asli (quality score 0-10)
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
df_raw[TARGET_COLUMN].value_counts().sort_index().plot(kind='bar', color='skyblue')
plt.title('Distribusi Quality Score (Asli)')
plt.xlabel('Quality Score')
plt.ylabel('Count')
plt.xticks(rotation=0)

# Visualisasi distribusi target biner (good vs bad)
plt.subplot(1, 2, 2)
binary_target = (df_raw[TARGET_COLUMN] >= 6).astype(int)
binary_counts = binary_target.value_counts()
plt.bar(['Bad (< 6)', 'Good (>= 6)'], [binary_counts[0], binary_counts[1]], color=['coral', 'lightgreen'])
plt.title('Distribusi Target Biner')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print(f"\nTarget balance (binary):")
print(f"Bad (< 6): {binary_counts[0]} ({binary_counts[0] / len(df_raw) * 100:.1f}%)")
print(f"Good (>= 6): {binary_counts[1]} ({binary_counts[1] / len(df_raw) * 100:.1f}%)")

**4.6 Visualisasi**: korelasi antar fitur

In [ ]:
# Heatmap korelasi
plt.figure(figsize=(12, 10))
correlation_matrix = df_raw.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap - Wine Quality Features')
plt.tight_layout()
plt.show()

# Fitur dengan korelasi tertinggi terhadap quality
quality_corr = correlation_matrix[TARGET_COLUMN].sort_values(ascending=False)
print("\nKorelasi fitur dengan quality (dari tertinggi):")
print(quality_corr)

**4.7 Visualisasi**: distribusi beberapa fitur penting

In [ ]:
# Visualisasi distribusi fitur-fitur penting
important_features = ['alcohol', 'density', 'volatile acidity', 'chlorides']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, feature in enumerate(important_features):
    axes[idx].hist(df_raw[feature], bins=30, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribusi {feature}')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# **5. Data Preprocessing**

Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

Jika Anda menggunakan data teks, data mentah sering kali mengandung nilai kosong, duplikasi, atau rentang nilai yang tidak konsisten, yang dapat memengaruhi kinerja model. Oleh karena itu, proses ini bertujuan untuk membersihkan dan mempersiapkan data agar analisis berjalan optimal.

Berikut adalah tahapan-tahapan yang bisa dilakukan, tetapi **tidak terbatas** pada:
1. Menghapus atau Menangani Data Kosong (Missing Values)
2. Menghapus Data Duplikat
3. Normalisasi atau Standarisasi Fitur
4. Deteksi dan Penanganan Outlier
5. Encoding Data Kategorikal
6. Binning (Pengelompokan Data)

Cukup sesuaikan dengan karakteristik data yang kamu gunakan yah. Khususnya ketika kami menggunakan data tidak terstruktur.

**5.1 Preprocessing Manual** menggunakan langkah-langkah yang sama dengan script automation

In [ ]:
# Buat salinan dataframe untuk preprocessing
df_processed = df_raw.copy()

print("=== PREPROCESSING STEPS ===")
print(f"\n1. Data awal: {df_processed.shape[0]} baris x {df_processed.shape[1]} kolom")

# Step 1: Pastikan semua kolom numerik
for col in df_processed.columns:
    df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')
print(f"\n2. Setelah konversi numerik: {df_processed.shape[0]} baris")

# Step 2: Binarisasi target (quality >= 6 -> 1, < 6 -> 0)
original_quality = df_processed[TARGET_COLUMN].copy()
df_processed[TARGET_COLUMN] = (df_processed[TARGET_COLUMN].fillna(0) >= 6).astype(int)
print(f"\n3. Target dibinarisasi:")
print(f"   - Good (>= 6): {(df_processed[TARGET_COLUMN] == 1).sum()}")
print(f"   - Bad (< 6): {(df_processed[TARGET_COLUMN] == 0).sum()}")

# Step 3: Imputasi median untuk missing values (jika ada)
feature_cols = [c for c in df_processed.columns if c != TARGET_COLUMN]
missing_before = df_processed[feature_cols].isna().sum().sum()
for col in feature_cols:
    median = df_processed[col].median()
    if pd.isna(median):
        median = 0.0
    df_processed[col] = df_processed[col].fillna(median)
missing_after = df_processed.isna().sum().sum()
print(f"\n4. Missing values: {missing_before} -> {missing_after}")

# Step 4: Drop duplikat
n_before_drop = len(df_processed)
df_processed = df_processed.drop_duplicates().reset_index(drop=True)
n_after_drop = len(df_processed)
print(f"\n5. Drop duplikat: {n_before_drop} -> {n_after_drop} baris (removed {n_before_drop - n_after_drop})")

# Step 5: Pastikan semua kolom numerik
df_processed = df_processed.apply(pd.to_numeric)

print(f"\n=== HASIL PREPROCESSING ===")
print(f"Shape final: {df_processed.shape[0]} baris x {df_processed.shape[1]} kolom")
print(f"Missing values: {df_processed.isna().sum().sum()}")
print(f"Duplikat: {df_processed.duplicated().sum()}")
print(f"\nTarget balance:")
print(df_processed[TARGET_COLUMN].value_counts())

**5.2 Verifikasi hasil preprocessing** dengan menggunakan fungsi dari script automation

In [ ]:
# Verifikasi bahwa hasil manual sama dengan hasil dari fungsi automation
df_automated = preprocess_dataset(df_raw, output_path=None)

print("=== VERIFIKASI ===")
print(f"Shape manual   : {df_processed.shape}")
print(f"Shape automated: {df_automated.shape}")
print(f"\nApakah hasil identik? {df_processed.equals(df_automated)}")

if not df_processed.equals(df_automated):
    print("\nPerbedaan ditemukan:")
    for col in df_processed.columns:
        if not df_processed[col].equals(df_automated[col]):
            print(f"  - Kolom '{col}' berbeda")

**5.3 Simpan hasil preprocessing**

In [ ]:
# Simpan dataset hasil preprocessing
df_processed.to_csv(OUTPUT_PATH, index=False)
print(f"Dataset preprocessing berhasil disimpan ke: {OUTPUT_PATH}")
print(f"\nRingkasan:")
print(f"- Baris: {len(df_processed)}")
print(f"- Kolom: {len(df_processed.columns)}")
print(f"- Missing: {df_processed.isna().sum().sum()}")
print(f"- Target balance: {df_processed[TARGET_COLUMN].value_counts().to_dict()}")

**5.4 Visualisasi perbandingan sebelum dan sesudah preprocessing**

In [ ]:
# Perbandingan distribusi target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sebelum (quality score asli)
axes[0].hist(original_quality, bins=range(0, 11), edgecolor='black', alpha=0.7, color='skyblue')
axes[0].set_title('Target Sebelum Preprocessing\n(Quality Score 0-10)')
axes[0].set_xlabel('Quality Score')
axes[0].set_ylabel('Count')
axes[0].set_xticks(range(0, 11))

# Sesudah (binary: 0=bad, 1=good)
counts = df_processed[TARGET_COLUMN].value_counts().sort_index()
axes[1].bar(['Bad (< 6)', 'Good (>= 6)'], [counts[0], counts[1]], color=['coral', 'lightgreen'], edgecolor='black')
axes[1].set_title('Target Sesudah Preprocessing\n(Binary Classification)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print("\n=== DATASET SIAP UNTUK MODELLING ===")